# 02 — Modeling: Baseline → Random Forest → XGBoost

Core question: **Which model + imbalance strategy gives the best PR-AUC?**

Imbalance strategies tested:
- class_weight='balanced' (LR + RF)
- SMOTE oversampling
- scale_pos_weight (XGBoost)
- Combined: SMOTE + XGBoost

Key metric: **PR-AUC** (not accuracy — see notebook 01 for why)

Run time: ~5–10 minutes for all models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, classification_report
import sys; sys.path.insert(0, '..')
from src.features import load_raw, build_features, get_X_y
from src.model import FailureClassifier, find_cost_optimal_threshold

%matplotlib inline
sns.set_theme(style='darkgrid')

In [ ]:
df = build_features(load_raw())
X, y = get_X_y(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {y_train.shape[0]:,} | Test: {y_test.shape[0]:,}")
print(f"Train failure rate: {y_train.mean()*100:.1f}% | Test: {y_test.mean()*100:.1f}%")

In [ ]:
# Baseline: Logistic Regression with class_weight='balanced'
lr_clf = FailureClassifier(model_type='logistic', use_smote=False)
lr_clf.fit(X_train, y_train)
lr_results = lr_clf.evaluate(X_test, y_test)
print("Logistic Regression (baseline):")
print(lr_results)

In [ ]:
# Random Forest with SMOTE
rf_clf = FailureClassifier(model_type='rf', use_smote=True)
rf_clf.fit(X_train, y_train)
rf_results = rf_clf.evaluate(X_test, y_test)
print("Random Forest + SMOTE:")
print(rf_results)

In [ ]:
# XGBoost (scale_pos_weight + SMOTE)
xgb_clf = FailureClassifier(model_type='xgb', use_smote=True)
xgb_clf.fit(X_train, y_train)
xgb_results = xgb_clf.evaluate(X_test, y_test)
print("XGBoost + SMOTE:")
print(xgb_results)

In [ ]:
# Comparison table
results_df = pd.DataFrame([
    {'Model': 'LR (baseline)', **{k: v for k,v in lr_results.items() if k in ['roc_auc','pr_auc','f1','recall','precision','business_cost']}},
    {'Model': 'RF + SMOTE',   **{k: v for k,v in rf_results.items() if k in ['roc_auc','pr_auc','f1','recall','precision','business_cost']}},
    {'Model': 'XGBoost + SMOTE', **{k: v for k,v in xgb_results.items() if k in ['roc_auc','pr_auc','f1','recall','precision','business_cost']}},
])
print(results_df.to_string(index=False))
# NOTE: save this table — it goes in README.md

In [ ]:
# Save best model
xgb_clf.save('../models/')
print("Model saved. Proceed to notebook 03 for threshold tuning.")